In [2]:
# standardní knihovny Pythonu
import sys
import time
import re
from pathlib import Path
from urllib.parse import urljoin

# knihovny třetích stran
import pandas as pd
import requests
from bs4 import BeautifulSoup

# knihovny pro dynamický web scraping
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# nastavení root složky projektu pro import utility modulů
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

# sdílené konstanty a společné schéma datasetu
from utils.constants import HEADERS
from utils.schema import FINAL_COLUMNS

# základní technické čištění textu
from utils.text_cleaning import (
    clean_base_text,
    join_text_parts
)

In [3]:
# URL katalogu exchange předmětů Queen's University
LIST_URL = (
    "https://www.queensu.ca/academic-calendar/"
    "business/bachelor-commerce/courses-of-instruction/exchange/#falltext"
)

# metadata společná pro všechny záznamy datasetu
ACADEMIC_YEAR = "2025-2026"
UNIVERSITY = "Queen's University"
COUNTRY = "Canada"
CREDIT_SYSTEM = "Queen's units"
STUDY_LEVEL = "undergraduate"

In [5]:
# parsování seznamu předmětů z HTML katalogu
def parse_course_list(html):

    soup = BeautifulSoup(html, "html.parser")

    courses = []

    for semester_id, semester_label, semester_normalized in [
        ("falltextcontainer", "Fall", "winter"),
        ("wintertextcontainer", "Winter", "summer")
    ]:

        container = soup.find("div", id=semester_id)

        if not container:
            print(f"Nenalezen kontejner: {semester_id}")
            continue

        table = container.find("table", class_="sc_courselist")

        if not table:
            print(f"Nenalezena tabulka v: {semester_id}")
            continue

        for row in table.find_all("tr"):

            code_cell = row.find("td", class_="codecol")

            if not code_cell:
                continue

            code_link = code_cell.find("a", class_="bubblelink")

            if not code_link:
                continue

            course_code = clean_base_text(
                code_link.get_text()
            )

            detail_url = urljoin(
                LIST_URL,
                code_link["href"]
            )

            cells = row.find_all("td")

            if len(cells) < 2:
                continue

            title_cell = cells[1]

            # odstranění poznámek typu "* by permission only"
            for sup in title_cell.find_all("sup"):
                sup.decompose()

            # odstranění vnořených odkazů na ekvivalentní předměty
            for a in title_cell.find_all("a"):
                a.decompose()

            course_title = clean_base_text(
                title_cell.get_text()
            )

            hours_cell = row.find("td", class_="hourscol")

            credits_raw = (
                clean_base_text(hours_cell.get_text())
                if hours_cell
                else pd.NA
            )

            credits = (
                float(credits_raw)
                if not pd.isna(credits_raw)
                else pd.NA
            )

            courses.append({
                "course_title": course_title,
                "course_code": course_code,
                "university": UNIVERSITY,
                "country": COUNTRY,
                "source_url": detail_url,
                "academic_year": ACADEMIC_YEAR,
                "semester": semester_label,
                "semester_normalized": semester_normalized,
                "study_level": STUDY_LEVEL,
                "credits": credits,
                "credit_system": CREDIT_SYSTEM,

                # orientační převod podle doporučení OZS:
                # 1 Queen's unit = 2 ECTS
                "ects_credits": (
                    credits * 2
                    if not pd.isna(credits)
                    else pd.NA
                )
            })

    return courses

In [7]:
# inicializace Selenium driveru
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)

# načtení dynamicky vykresleného seznamu předmětů
driver.get(LIST_URL)
time.sleep(10)

html = driver.page_source

In [8]:
# parsování seznamu předmětů
courses = parse_course_list(html)

print(f"Nalezeno kurzů: {len(courses)}")

Nalezeno kurzů: 103


In [9]:
# parsování detailní stránky předmětu
def parse_course_detail(detail_html):

    soup = BeautifulSoup(detail_html, "html.parser")

    courseblock = soup.select_one(".courseblock")

    if courseblock is None:
        raise ValueError("Course detail not found")

    description_el = courseblock.select_one(".courseblockextra")

    description = (
        clean_base_text(description_el.get_text(" "))
        if description_el
        else pd.NA
    )

    return {
        "aims": description,
        "course_content": pd.NA,
        "learning_outcomes": pd.NA,
        "assessment_methods": pd.NA,
        "literature": pd.NA,
    }

In [11]:
# stažení a parsování detailních stránek předmětů
full_courses = []

for i, course in enumerate(courses, start=1):

    print(f"{i}/{len(courses)} - {course['course_code']}")

    driver.get(course["source_url"])

    # respektování crawl-delay uvedeného v robots.txt
    time.sleep(10)

    detail_html = driver.page_source
    detail_data = parse_course_detail(detail_html)

    full_courses.append(
        course | detail_data
    )

1/103 - COMM 163
2/103 - COMM 200
3/103 - COMM 201
4/103 - COMM 211
5/103 - COMM 212
6/103 - COMM 221
7/103 - COMM 231
8/103 - COMM 251
9/103 - COMM 301
10/103 - COMM 305
11/103 - COMM 306
12/103 - COMM 309
13/103 - COMM 311
14/103 - COMM 312
15/103 - COMM 313
16/103 - COMM 317
17/103 - COMM 319
18/103 - COMM 321
19/103 - COMM 322
20/103 - COMM 323
21/103 - COMM 324
22/103 - COMM 326
23/103 - COMM 329
24/103 - COMM 333
25/103 - COMM 338
26/103 - COMM 339
27/103 - COMM 341
28/103 - COMM 351
29/103 - COMM 353
30/103 - COMM 356
31/103 - COMM 357
32/103 - COMM 358
33/103 - COMM 370
34/103 - COMM 373
35/103 - COMM 374
36/103 - COMM 376
37/103 - COMM 381
38/103 - COMM 385
39/103 - COMM 392
40/103 - COMM 394
41/103 - COMM 395
42/103 - COMM 401
43/103 - COMM 404
44/103 - COMM 405
45/103 - COMM 406
46/103 - COMM 408
47/103 - COMM 410
48/103 - COMM 411
49/103 - COMM 414
50/103 - COMM 416
51/103 - COMM 434
52/103 - COMM 495
53/103 - COMM 122
54/103 - COMM 132
55/103 - COMM 172
56/103 - COMM 190
5

In [12]:
# vytvoření dataframe s naparsovanými předměty
df = pd.DataFrame(full_courses)

# seřazení sloupců podle společného schématu datasetu
df = df[FINAL_COLUMNS]

print("Celkem získaných předmětů:", len(df))

Celkem získaných předmětů: 103


In [15]:
# export naparsovaných sylabů Queen's University do interim vrstvy
OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "queens_syllabi_raw_parsed.csv"
)

df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(f"Uloženo {len(df)} kurzů do {OUTPUT_PATH.name}")

Uloženo 103 kurzů do queens_syllabi_raw_parsed.csv


In [16]:
# ukončení Selenium driveru
driver.quit()